# House Price Predictor using AWS Services

This notebook implements an end-to-end Machine Learning workflow for predicting house prices using AWS services.

### AWS Services Used
- Amazon S3
- AWS Glue Data Catalog
- Amazon Athena
- Amazon SageMaker Feature Store
- Amazon SageMaker Model Registry

### ML Algorithm
Random Forest Regressor (scikit-learn)

> Model training is performed locally to minimize AWS compute costs.

In [5]:
# ===============================
# Standard Python Libraries
# ===============================
import os
import io
import json
import time
import uuid
import tarfile
import pickle
from datetime import datetime, timezone

# ===============================
# Data Processing
# ===============================
import numpy as np
import pandas as pd

# ===============================
# Machine Learning
# ===============================
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ===============================
# AWS SDK
# ===============================
#!pip install boto3 pandas numpy scikit-learn
import boto3
import botocore

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


## AWS Configuration

This section:

- Detects the configured AWS Region
- Retrieves AWS Account ID
- Creates reusable boto3 clients
- Generates unique names for this lab

In [37]:
# ======================================
# Detect AWS Region
# ======================================

import os
import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.Session().region_name
    or "us-east-1"
)

# ======================================
# Create AWS Session
# ======================================

session = boto3.Session(region_name=REGION)

# ======================================
# AWS Clients
# ======================================

sts = session.client("sts")
s3 = session.client("s3")
glue = session.client("glue")
athena = session.client("athena")
sagemaker = session.client("sagemaker")

# ======================================
# Verify Identity
# ======================================

featurestore_runtime = session.client(
    "sagemaker-featurestore-runtime"
)

identity = sts.get_caller_identity()

ACCOUNT_ID = identity["Account"]
USER_ARN = identity["Arn"]

print("=" * 60)
print("AWS Configuration")
print("=" * 60)
print(f"Region     : {REGION}")
print(f"Account ID : {ACCOUNT_ID}")
print(f"IAM User   : {USER_ARN.split('/')[-1]}")
print("=" * 60)

AWS Configuration
Region     : us-east-1
Account ID : 483955931464
IAM User   : student


# Step 1 - Create S3 Bucket

This section creates an S3 bucket that will be used to store:

- Raw dataset
- Athena query results
- Feature Store offline data
- Model artifacts

In [7]:
# ======================================
# Project Configuration
# ======================================

PROJECT_NAME = "house-price-predictor"

LAB_ID = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

BUCKET_NAME = f"houselab-{ACCOUNT_ID}-{REGION}"

DATABASE_NAME = "houselab_db"

TABLE_NAME = "houses_raw"

FEATURE_GROUP_NAME = f"house-price-features-{LAB_ID}"

MODEL_PACKAGE_GROUP = "house-price-model-group"

print("Project Configuration")
print("-" * 50)
print(f"Bucket Name         : {BUCKET_NAME}")
print(f"Glue Database       : {DATABASE_NAME}")
print(f"Glue Table          : {TABLE_NAME}")
print(f"Feature Group       : {FEATURE_GROUP_NAME}")
print(f"Model Package Group : {MODEL_PACKAGE_GROUP}")

Project Configuration
--------------------------------------------------
Bucket Name         : houselab-483955931464-us-east-1
Glue Database       : houselab_db
Glue Table          : houses_raw
Feature Group       : house-price-features-20260804062800
Model Package Group : house-price-model-group


In [8]:
print("=" * 60)
print("Creating S3 Bucket")
print("=" * 60)

try:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=BUCKET_NAME)
    else:
        s3.create_bucket(
            Bucket=BUCKET_NAME,
            CreateBucketConfiguration={
                "LocationConstraint": REGION
            }
        )

    print(f"✅ Bucket '{BUCKET_NAME}' created successfully.")

except botocore.exceptions.ClientError as e:
    error_code = e.response["Error"]["Code"]

    if error_code == "BucketAlreadyOwnedByYou":
        print(f"ℹ️ Bucket '{BUCKET_NAME}' already exists.")

    else:
        raise

Creating S3 Bucket
✅ Bucket 'houselab-483955931464-us-east-1' created successfully.


In [9]:
print("=" * 60)
print("Configuring Bucket Security")
print("=" * 60)

# Block all public access
s3.put_public_access_block(
    Bucket=BUCKET_NAME,
    PublicAccessBlockConfiguration={
        "BlockPublicAcls": True,
        "IgnorePublicAcls": True,
        "BlockPublicPolicy": True,
        "RestrictPublicBuckets": True
    }
)

# Enable default encryption
s3.put_bucket_encryption(
    Bucket=BUCKET_NAME,
    ServerSideEncryptionConfiguration={
        "Rules": [
            {
                "ApplyServerSideEncryptionByDefault": {
                    "SSEAlgorithm": "AES256"
                }
            }
        ]
    }
)

print("✅ Public access blocked")
print("✅ Server-side encryption enabled")

Configuring Bucket Security
✅ Public access blocked
✅ Server-side encryption enabled


In [10]:
response = s3.list_buckets()

print("\nAvailable Buckets:\n")

for bucket in response["Buckets"]:
    if bucket["Name"] == BUCKET_NAME:
        print(f"✅ {bucket['Name']}")


Available Buckets:

✅ houselab-483955931464-us-east-1


# Step 2 - Generate Synthetic House Dataset

Generate a synthetic dataset of 500 residential properties.

The target variable (`price_usd`) is calculated using the formula provided in the assignment with Gaussian noise to simulate real-world market variations.

In [13]:
print("=" * 60)
print("Generating Synthetic House Dataset")
print("=" * 60)

# Ensure reproducibility
np.random.seed(42)

NUM_RECORDS = 500

houses_df = pd.DataFrame({
    "size_sqft": np.random.randint(600, 3000, NUM_RECORDS),
    "bedrooms": np.random.randint(1, 6, NUM_RECORDS),
    "age_years": np.random.randint(0, 50, NUM_RECORDS),
    "distance_km": np.round(np.random.uniform(1.0, 30.0, NUM_RECORDS), 2),
    "has_garage": np.random.randint(0, 2, NUM_RECORDS)
})

# Generate house price using the assignment formula
noise = np.random.normal(0, 30000, NUM_RECORDS)

houses_df["price_usd"] = (
    (150 * houses_df["size_sqft"])
    + (20000 * houses_df["bedrooms"])
    - (1500 * houses_df["age_years"])
    - (8000 * houses_df["distance_km"])
    + (25000 * houses_df["has_garage"])
    + noise
)

# Clip values between 50K and 2M
houses_df["price_usd"] = (
    houses_df["price_usd"]
    .clip(50000, 2000000)
    .round(-3)
    .astype(int)
)

print(f"Dataset created successfully with {len(houses_df)} records.")

Generating Synthetic House Dataset
Dataset created successfully with 500 records.


In [14]:
print("=" * 60)
print("Dataset Preview")
print("=" * 60)

houses_df.head(10)

Dataset Preview


,size_sqft,bedrooms,age_years,distance_km,has_garage,price_usd
0,1460,5,27,4.74,0,216000
1,1894,1,49,10.72,1,107000
2,1730,4,20,22.56,0,147000
3,1695,1,48,5.66,1,192000
4,2238,1,6,24.72,0,98000
5,2769,5,16,25.13,0,334000
6,1066,4,19,15.72,1,161000
7,1838,4,40,1.19,1,289000
8,930,4,48,9.32,0,108000
9,2082,3,19,18.89,1,200000


In [15]:
print("=" * 60)
print("Dataset Information")
print("=" * 60)

houses_df.info()

Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   size_sqft    500 non-null    int32  
 1   bedrooms     500 non-null    int32  
 2   age_years    500 non-null    int32  
 3   distance_km  500 non-null    float64
 4   has_garage   500 non-null    int32  
 5   price_usd    500 non-null    int64  
dtypes: float64(1), int32(4), int64(1)
memory usage: 15.8 KB


In [16]:
print("=" * 60)
print("Statistical Summary")
print("=" * 60)

houses_df.describe().round(2)

Statistical Summary


,size_sqft,bedrooms,age_years,distance_km,has_garage,price_usd
count,500.00,500.00,500.00,500.00,500.00,500.00
mean,1827.22,2.99,25.43,15.47,0.52,195268.00
std,660.89,1.44,14.38,8.40,0.50,114921.76
min,601.00,1.00,0.00,1.14,0.00,50000.00
25%,1282.50,2.00,13.00,7.89,0.00,89000.00
50%,1810.50,3.00,25.50,15.94,1.00,183000.00
75%,2358.25,4.00,38.00,22.20,1.00,286500.00
max,2993.00,5.00,49.00,29.98,1.00,496000.00


In [17]:
print("=" * 60)
print("Data Quality Checks")
print("=" * 60)

print("Missing Values")
print(houses_df.isnull().sum())

print("\nDuplicate Rows")
print(houses_df.duplicated().sum())

Data Quality Checks
Missing Values
size_sqft      0
bedrooms       0
age_years      0
distance_km    0
has_garage     0
price_usd      0
dtype: int64

Duplicate Rows
0


# Step 3 - Upload Dataset to Amazon S3

Convert the generated DataFrame into a CSV file and upload it to the S3 bucket.

S3 Path:

s3://<bucket-name>/houselab/raw/houses.csv

In [18]:
print("=" * 60)
print("Preparing S3 Upload")
print("=" * 60)

RAW_FOLDER = "houselab/raw/"
CSV_FILE_NAME = "houses.csv"

S3_OBJECT_KEY = f"{RAW_FOLDER}{CSV_FILE_NAME}"

print(f"S3 Bucket : {BUCKET_NAME}")
print(f"S3 Key    : {S3_OBJECT_KEY}")

Preparing S3 Upload
S3 Bucket : houselab-483955931464-us-east-1
S3 Key    : houselab/raw/houses.csv


In [19]:
print("=" * 60)
print("Uploading Dataset to S3")
print("=" * 60)

csv_buffer = io.StringIO()

houses_df.to_csv(csv_buffer, index=False)

s3.put_object(
    Bucket=BUCKET_NAME,
    Key=S3_OBJECT_KEY,
    Body=csv_buffer.getvalue()
)

print("✅ Dataset uploaded successfully!")
print(f"s3://{BUCKET_NAME}/{S3_OBJECT_KEY}")

Uploading Dataset to S3
✅ Dataset uploaded successfully!
s3://houselab-483955931464-us-east-1/houselab/raw/houses.csv


In [20]:
print("=" * 60)
print("Verifying Uploaded File")
print("=" * 60)

response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=RAW_FOLDER
)

if "Contents" in response:
    for obj in response["Contents"]:
        print(f"✅ {obj['Key']} ({obj['Size']} bytes)")
else:
    print("No files found.")

Verifying Uploaded File
✅ houselab/raw/houses.csv (12564 bytes)


# Step 4 - Create AWS Glue Data Catalog

Create a Glue Database and register the uploaded CSV as an external table.

This enables Amazon Athena to query the data stored in S3.

In [21]:
print("=" * 60)
print("Creating Glue Database")
print("=" * 60)

try:
    glue.create_database(
        DatabaseInput={
            "Name": DATABASE_NAME,
            "Description": "House Price Prediction Database"
        }
    )

    print(f"✅ Database '{DATABASE_NAME}' created.")

except glue.exceptions.AlreadyExistsException:
    print(f"ℹ️ Database '{DATABASE_NAME}' already exists.")

Creating Glue Database
✅ Database 'houselab_db' created.


In [22]:
print("=" * 60)
print("Creating Glue Table")
print("=" * 60)

glue_columns = [
    {"Name": "size_sqft", "Type": "int"},
    {"Name": "bedrooms", "Type": "int"},
    {"Name": "age_years", "Type": "int"},
    {"Name": "distance_km", "Type": "double"},
    {"Name": "has_garage", "Type": "int"},
    {"Name": "price_usd", "Type": "int"},
]

try:

    glue.create_table(
        DatabaseName=DATABASE_NAME,
        TableInput={
            "Name": TABLE_NAME,
            "Description": "House dataset stored in Amazon S3",
            "TableType": "EXTERNAL_TABLE",

            "Parameters": {
                "classification": "csv",
                "skip.header.line.count": "1"
            },

            "StorageDescriptor": {

                "Columns": glue_columns,

                "Location": f"s3://{BUCKET_NAME}/{RAW_FOLDER}",

                "InputFormat":
                    "org.apache.hadoop.mapred.TextInputFormat",

                "OutputFormat":
                    "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",

                "SerdeInfo": {

                    "SerializationLibrary":
                        "org.apache.hadoop.hive.serde2.OpenCSVSerde",

                    "Parameters": {
                        "separatorChar": ",",
                        "quoteChar": "\"",
                        "escapeChar": "\\"
                    }
                }
            }
        }
    )

    print(f"✅ Table '{TABLE_NAME}' created.")

except glue.exceptions.AlreadyExistsException:
    print(f"ℹ️ Table '{TABLE_NAME}' already exists.")

Creating Glue Table
✅ Table 'houses_raw' created.


In [23]:
print("=" * 60)
print("Verifying Glue Table")
print("=" * 60)

table = glue.get_table(
    DatabaseName=DATABASE_NAME,
    Name=TABLE_NAME
)

print("Database :", DATABASE_NAME)
print("Table    :", table["Table"]["Name"])
print("Location :", table["Table"]["StorageDescriptor"]["Location"])

Verifying Glue Table
Database : houselab_db
Table    : houses_raw
Location : s3://houselab-483955931464-us-east-1/houselab/raw/


# Step 5 - Query Data using Amazon Athena

Execute an SQL query on the uploaded CSV stored in Amazon S3 using Amazon Athena.

The query calculates the average house price grouped by the number of bedrooms.

In [24]:
print("=" * 60)
print("Preparing Athena Output Location")
print("=" * 60)

ATHENA_OUTPUT = f"s3://{BUCKET_NAME}/houselab/athena-results/"

print("Athena Output Location:")
print(ATHENA_OUTPUT)

Preparing Athena Output Location
Athena Output Location:
s3://houselab-483955931464-us-east-1/houselab/athena-results/


In [25]:
print("=" * 60)
print("Executing Athena Query")
print("=" * 60)

query = f"""
SELECT
    bedrooms,
    AVG(CAST(price_usd AS DOUBLE)) AS avg_price
FROM {TABLE_NAME}
GROUP BY bedrooms
ORDER BY bedrooms
"""

response = athena.start_query_execution(
    QueryString=query,
    QueryExecutionContext={
        "Database": DATABASE_NAME
    },
    ResultConfiguration={
        "OutputLocation": ATHENA_OUTPUT
    }
)

query_execution_id = response["QueryExecutionId"]

print("Query Execution ID:")
print(query_execution_id)

Executing Athena Query
Query Execution ID:
6c64f0c7-4f4c-4418-9065-7956deb0c6a6


In [26]:
print("=" * 60)
print("Waiting for Athena Query")
print("=" * 60)

while True:

    response = athena.get_query_execution(
        QueryExecutionId=query_execution_id
    )

    state = response["QueryExecution"]["Status"]["State"]

    print(f"Current State : {state}")

    if state in ["SUCCEEDED", "FAILED", "CANCELLED"]:
        break

    time.sleep(2)

print(f"\nFinal Status : {state}")

Waiting for Athena Query
Current State : SUCCEEDED

Final Status : SUCCEEDED


In [27]:
print("=" * 60)
print("Reading Athena Results")
print("=" * 60)

results = athena.get_query_results(
    QueryExecutionId=query_execution_id
)

rows = []

for row in results["ResultSet"]["Rows"]:

    values = []

    for value in row["Data"]:
        values.append(value.get("VarCharValue"))

    rows.append(values)

athena_df = pd.DataFrame(
    rows[1:],
    columns=rows[0]
)

athena_df

Reading Athena Results


,bedrooms,avg_price
0,1,162691.58878504674
1,2,170393.6170212766
2,3,195840.0
3,4,209462.36559139786
4,5,237216.98113207548


In [28]:
print("=" * 60)
print("Athena Result Files")
print("=" * 60)

response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix="houselab/athena-results/"
)

for obj in response.get("Contents", []):
    print(f"✅ {obj['Key']}")

Athena Result Files
✅ houselab/athena-results/6c64f0c7-4f4c-4418-9065-7956deb0c6a6.csv
✅ houselab/athena-results/6c64f0c7-4f4c-4418-9065-7956deb0c6a6.csv.metadata


# Step 6 - Prepare Dataset for SageMaker Feature Store

Feature Store requires:

- record_id (unique identifier)
- event_time (timestamp)

These columns are added before ingesting records.

In [29]:
print("=" * 60)
print("Preparing Feature Store Dataset")
print("=" * 60)

feature_df = houses_df.copy()

# Unique record identifier
feature_df["record_id"] = [
    str(uuid.uuid4()) for _ in range(len(feature_df))
]

# Event time (ISO 8601 format)
event_time = datetime.now(timezone.utc).isoformat()

feature_df["event_time"] = event_time

# Reorder columns
feature_df = feature_df[
    [
        "record_id",
        "event_time",
        "size_sqft",
        "bedrooms",
        "age_years",
        "distance_km",
        "has_garage",
        "price_usd",
    ]
]

print("Feature Store dataset prepared.")
feature_df.head()

Preparing Feature Store Dataset
Feature Store dataset prepared.


,record_id,event_time,size_sqft,bedrooms,age_years,distance_km,has_garage,price_usd
0,c94f93a1-e634-47a6-8c21-ad1284b3a16b,2026-08-04T16:15:45.487651+00:00,1460,5,27,4.74,0,216000
1,492e6ca2-7c8a-4dde-a96d-e011b763f1d9,2026-08-04T16:15:45.487651+00:00,1894,1,49,10.72,1,107000
2,a0608a30-6f55-4ca7-8e0c-412fb161f086,2026-08-04T16:15:45.487651+00:00,1730,4,20,22.56,0,147000
3,92a6bf5b-fb9c-4812-912d-585f15315d7f,2026-08-04T16:15:45.487651+00:00,1695,1,48,5.66,1,192000
4,f0aa72b5-5d2e-4b9f-8c8f-2b2fc62efde3,2026-08-04T16:15:45.487651+00:00,2238,1,6,24.72,0,98000


In [30]:
print("=" * 60)
print("Defining Feature Group")
print("=" * 60)

feature_definitions = [
    {"FeatureName": "record_id", "FeatureType": "String"},
    {"FeatureName": "event_time", "FeatureType": "String"},
    {"FeatureName": "size_sqft", "FeatureType": "Integral"},
    {"FeatureName": "bedrooms", "FeatureType": "Integral"},
    {"FeatureName": "age_years", "FeatureType": "Integral"},
    {"FeatureName": "distance_km", "FeatureType": "Fractional"},
    {"FeatureName": "has_garage", "FeatureType": "Integral"},
    {"FeatureName": "price_usd", "FeatureType": "Integral"},
]

print("Feature definitions created.")

Defining Feature Group
Feature definitions created.


In [32]:
print("=" * 60)
print("Configuring SageMaker Execution Role")
print("=" * 60)

SAGEMAKER_ROLE_ARN = (
    "arn:aws:iam::483955931464:role/"
    "service-role/AmazonSageMakerAdminIAMExecutionRole"
)

print("Execution Role:")
print(SAGEMAKER_ROLE_ARN)

Configuring SageMaker Execution Role
Execution Role:
arn:aws:iam::483955931464:role/service-role/AmazonSageMakerAdminIAMExecutionRole


In [33]:
print("=" * 60)
print("Creating Feature Group")
print("=" * 60)

try:

    sagemaker.create_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME,

        RecordIdentifierFeatureName="record_id",

        EventTimeFeatureName="event_time",

        FeatureDefinitions=feature_definitions,

        OnlineStoreConfig={
            "EnableOnlineStore": True
        },

        OfflineStoreConfig={
            "S3StorageConfig": {
                "S3Uri": f"s3://{BUCKET_NAME}/houselab/feature-store/"
            }
        },

        RoleArn=SAGEMAKER_ROLE_ARN

    )

    print("✅ Feature Group creation started.")

except sagemaker.exceptions.ResourceInUse:
    print("ℹ️ Feature Group already exists.")

Creating Feature Group
✅ Feature Group creation started.


In [34]:
print("=" * 60)
print("Waiting for Feature Group")
print("=" * 60)

while True:

    response = sagemaker.describe_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME
    )

    status = response["FeatureGroupStatus"]

    print(status)

    if status == "Created":
        break

    if status == "CreateFailed":
        raise Exception(response)

    time.sleep(10)

print("\n✅ Feature Group Ready")

Waiting for Feature Group
Created

✅ Feature Group Ready


# Step 7 - Ingest Records into SageMaker Feature Store

Insert all prepared house records into the online Feature Store.

In [35]:
print("=" * 60)
print("Preparing Records for Feature Store")
print("=" * 60)

feature_records = []

for _, row in feature_df.iterrows():

    record = []

    for column in feature_df.columns:

        value = row[column]

        record.append({
            "FeatureName": column,
            "ValueAsString": str(value)
        })

    feature_records.append(record)

print(f"Prepared {len(feature_records)} records.")

Preparing Records for Feature Store
Prepared 500 records.


In [38]:
print("=" * 60)
print("Ingesting Records")
print("=" * 60)

for i, record in enumerate(feature_records):

    featurestore_runtime.put_record(
        FeatureGroupName=FEATURE_GROUP_NAME,
        Record=record
    )

    if (i + 1) % 100 == 0:
        print(f"{i + 1} records inserted...")

print("\n✅ All records successfully ingested.")

Ingesting Records
100 records inserted...
200 records inserted...
300 records inserted...
400 records inserted...
500 records inserted...

✅ All records successfully ingested.


# Step 8 - Prepare Training Data

Separate input features and target variable, then split the dataset into training and testing sets.

In [39]:
print("=" * 60)
print("Preparing Training Dataset")
print("=" * 60)

X = houses_df.drop(columns=["price_usd"])
y = houses_df["price_usd"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"Training Samples : {len(X_train)}")
print(f"Testing Samples  : {len(X_test)}")

Preparing Training Dataset
Training Samples : 400
Testing Samples  : 100


# Step 9 - Train Random Forest Regressor

In [40]:
print("=" * 60)
print("Training Random Forest Model")
print("=" * 60)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("✅ Model training completed.")

Training Random Forest Model
✅ Model training completed.


# Step 10 - Predict House Prices

In [41]:
print("=" * 60)
print("Generating Predictions")
print("=" * 60)

predictions = model.predict(X_test)

print("Prediction completed.")

Generating Predictions
Prediction completed.


# Step 11 - Evaluate Model

In [42]:
print("=" * 60)
print("Model Evaluation")
print("=" * 60)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R²   : {r2:.4f}")

Model Evaluation
MAE  : 26,019.20
RMSE : 33,920.81
R²   : 0.9131


In [43]:
print("=" * 60)
print("Feature Importance")
print("=" * 60)

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

importance_df

Feature Importance


,Feature,Importance
0,size_sqft,0.638687
1,distance_km,0.272966
2,age_years,0.043274
3,bedrooms,0.036395
4,has_garage,0.008677


In [44]:
comparison_df = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": predictions.round(0).astype(int)
})

comparison_df.head(10)

,Actual Price,Predicted Price
0,326000,340100
1,359000,317200
2,237000,257040
3,210000,217080
4,271000,282940
5,402000,406020
6,263000,241070
7,247000,153480
8,140000,131540
9,206000,202130


# Step 12 - Package the Trained Model

Serialize the trained Random Forest model and package it as `model.tar.gz` for SageMaker Model Registry.

In [45]:
print("=" * 60)
print("Saving Trained Model")
print("=" * 60)

MODEL_DIR = "model_artifacts"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_FILE = os.path.join(MODEL_DIR, "model.pkl")

with open(MODEL_FILE, "wb") as f:
    pickle.dump(model, f)

print(f"✅ Model saved to: {MODEL_FILE}")

Saving Trained Model
✅ Model saved to: model_artifacts\model.pkl


# Step 12 - Package the Trained Model

Serialize the trained Random Forest model and package it as `model.tar.gz` for SageMaker Model Registry.

In [46]:
print("=" * 60)
print("Saving Trained Model")
print("=" * 60)

MODEL_DIR = "model_artifacts"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_FILE = os.path.join(MODEL_DIR, "model.pkl")

with open(MODEL_FILE, "wb") as f:
    pickle.dump(model, f)

print(f"✅ Model saved to: {MODEL_FILE}")

Saving Trained Model
✅ Model saved to: model_artifacts\model.pkl


In [47]:
print("=" * 60)
print("Creating model.tar.gz")
print("=" * 60)

MODEL_ARCHIVE = "model.tar.gz"

with tarfile.open(MODEL_ARCHIVE, "w:gz") as tar:
    tar.add(MODEL_FILE, arcname="model.pkl")

print(f"✅ Archive created: {MODEL_ARCHIVE}")

Creating model.tar.gz
✅ Archive created: model.tar.gz


In [48]:
print("=" * 60)
print("Uploading Model to S3")
print("=" * 60)

MODEL_S3_KEY = "houselab/model/model.tar.gz"

with open(MODEL_ARCHIVE, "rb") as data:
    s3.upload_fileobj(
        data,
        BUCKET_NAME,
        MODEL_S3_KEY
    )

MODEL_S3_URI = f"s3://{BUCKET_NAME}/{MODEL_S3_KEY}"

print("✅ Model uploaded successfully.")
print(MODEL_S3_URI)

Uploading Model to S3
✅ Model uploaded successfully.
s3://houselab-483955931464-us-east-1/houselab/model/model.tar.gz


In [49]:
print("=" * 60)
print("Verifying Model Upload")
print("=" * 60)

response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix="houselab/model/"
)

for obj in response.get("Contents", []):
    print(f"✅ {obj['Key']} ({obj['Size']} bytes)")

Verifying Model Upload
✅ houselab/model/model.tar.gz (538555 bytes)


# Step 13 - Register Model in SageMaker Model Registry

Create a Model Package Group and register the trained model artifact.

In [50]:
print("=" * 60)
print("Creating Model Package Group")
print("=" * 60)

try:
    sagemaker.create_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        ModelPackageGroupDescription="House Price Prediction Models"
    )

    print(f"✅ Model Package Group '{MODEL_PACKAGE_GROUP}' created.")

except sagemaker.exceptions.ResourceInUse:
    print(f"ℹ️ Model Package Group '{MODEL_PACKAGE_GROUP}' already exists.")

Creating Model Package Group
✅ Model Package Group 'house-price-model-group' created.


In [51]:
print("=" * 60)
print("Registering Model")
print("=" * 60)

response = sagemaker.create_model_package(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    ModelApprovalStatus="PendingManualApproval",

    InferenceSpecification={
        "Containers": [
            {
                "Image": "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3",
                "ModelDataUrl": MODEL_S3_URI
            }
        ],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"]
    }
)

MODEL_PACKAGE_ARN = response["ModelPackageArn"]

print("✅ Model registered successfully.")
print(MODEL_PACKAGE_ARN)

Registering Model
✅ Model registered successfully.
arn:aws:sagemaker:us-east-1:483955931464:model-package/house-price-model-group/1


In [52]:
print("=" * 60)
print("Waiting for Model Registration")
print("=" * 60)

while True:

    response = sagemaker.describe_model_package(
        ModelPackageName=MODEL_PACKAGE_ARN
    )

    status = response["ModelPackageStatus"]

    print(status)

    if status == "Completed":
        break

    if status == "Failed":
        raise Exception(response["FailureReason"])

    time.sleep(5)

print("\n✅ Model Package Ready")

Waiting for Model Registration
Completed

✅ Model Package Ready


In [53]:
print("=" * 60)
print("Model Package Details")
print("=" * 60)

response = sagemaker.describe_model_package(
    ModelPackageName=MODEL_PACKAGE_ARN
)

print("Model Package Group :", response["ModelPackageGroupName"])
print("Approval Status     :", response["ModelApprovalStatus"])
print("Model Status        :", response["ModelPackageStatus"])

Model Package Details
Model Package Group : house-price-model-group
Approval Status     : PendingManualApproval
Model Status        : Completed


In [54]:
print("=" * 60)
print("Creating Model Card")
print("=" * 60)

model_card = f"""# House Price Prediction Model Card

## Model Information

- Model Name: House Price Predictor
- Algorithm: Random Forest Regressor
- Framework: Scikit-learn
- Number of Trees: 100
- Random State: 42

## Dataset

Synthetic dataset containing 500 house records.

Features:
- size_sqft
- bedrooms
- age_years
- distance_km
- has_garage

Target:
- price_usd

## Training Configuration

- Train/Test Split: 80/20
- Training Samples: 400
- Test Samples: 100

## Model Performance

| Metric | Value |
|--------|-------|
| MAE | {mae:.2f} |
| RMSE | {rmse:.2f} |
| R² Score | {r2:.4f} |

## Feature Importance

1. size_sqft
2. distance_km
3. age_years
4. bedrooms
5. has_garage

## Model Status

PendingManualApproval

## Author

BITS Pilani Professional AI/ML Programme Lab

Generated Automatically
"""

with open("model_card.md", "w", encoding="utf-8") as f:
    f.write(model_card)

print("✅ model_card.md created successfully.")

Creating Model Card
✅ model_card.md created successfully.


In [55]:
print("=" * 60)
print("Uploading Model Card")
print("=" * 60)

MODEL_CARD_S3_KEY = "houselab/model/model_card.md"

s3.upload_file(
    "model_card.md",
    BUCKET_NAME,
    MODEL_CARD_S3_KEY
)

print("✅ Model Card uploaded successfully.")

print(f"s3://{BUCKET_NAME}/{MODEL_CARD_S3_KEY}")

Uploading Model Card
✅ Model Card uploaded successfully.
s3://houselab-483955931464-us-east-1/houselab/model/model_card.md


In [56]:
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix="houselab/model/"
)

print("=" * 60)
print("Model Folder Contents")
print("=" * 60)

for obj in response["Contents"]:
    print(obj["Key"])

Model Folder Contents
houselab/model/model.tar.gz
houselab/model/model_card.md
